In [1]:
import dap_prinz_green_jobs.analysis.ojo_analysis.process_ojo_green_measures as pg

from dap_prinz_green_jobs import PROJECT_DIR, analysis_config
import dap_prinz_green_jobs.utils.plotting as pt
from datetime import datetime
import os
import pandas as pd
import polars as pl
import numpy as np
import altair as alt

import ast

In [2]:
#save graphs
today = datetime.today().strftime('%y%m%d')
graph_dir = str(PROJECT_DIR / f"outputs/figures/green_jobs_explorer/{today}/")

if not os.path.exists(graph_dir):
    print(f"Creating {graph_dir} directory")
    os.makedirs(graph_dir)
else:
    print(f"{graph_dir} directory already exists")

/Users/elizabethgallagher/Code/dap_prinz_green_jobs/outputs/figures/green_jobs_explorer/241128 directory already exists


In [3]:
#alt disable max rows

alt.data_transformers.disable_max_rows()


green = pt.NESTA_COLOURS_DICT['green'] # skills
blue = pt.NESTA_COLOURS_DICT['aqua'] # occupation
purple = pt.NESTA_COLOURS_DICT['purple'] # industry
red = pt.NESTA_COLOURS_DICT['red']
grey = "#1a1a1aff" # industry

chart_width = 200
chart_height = 450
x_value = 96

## 0. Load data

In [7]:
measures_all_ads = pd.read_parquet(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/analysis/{analysis_config['analysis_files']['agg_soc_date_stamp']}/combined_green_measures_and_meta.parquet")

2024-12-02 17:36:05,690 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [210]:
green_skills_data = pd.read_parquet(
    f"s3://prinz-green-jobs/outputs/data/ojo_application/extracted_green_measures/{analysis_config['skills_date_stamp']}/{analysis_config['green_skills_exploded_name']}")

In [12]:
created_data = pd.read_parquet(
    f"{analysis_config['deduplicated_data_dir']}{analysis_config['dedupe_key_columns_name']}")

In [13]:
created_data["quarter"] = pd.PeriodIndex(created_data.created, freq='Q')
created_data['quarter'] = created_data['quarter'].astype(str)
created_data['quarter'] = created_data['quarter'].apply(lambda x: x.replace("Q", " Q"))

In [45]:
measures_df = measures_all_ads[
['job_id', 'NUM_SPLIT_ENTS', 'PROP_GREEN', 'GREEN TIMESHARE', 'SOC_2020_EXT', 'SOC_2020_EXT_name', 'SIC', 'INDUSTRY GHG PER UNIT EMISSIONS']
].merge(created_data[['id', 'created']], how='inner', left_on='job_id', right_on='id')

In [46]:
measures_df["quarter"] = pd.PeriodIndex(measures_df.created, freq='Q')

In [19]:
skills_data = pl.read_parquet(
    f"{analysis_config['deduplicated_data_dir']}latest_update_20241114_skills.parquet",
    columns = ['id', 'esco_id']
)

In [20]:
skills_data = skills_data.with_columns(
    pl.col("id").cast(pl.Int64).alias("id"),
)

In [21]:
skills_data_time = skills_data.join(pl.from_pandas(created_data[['id', 'quarter']]), how='inner', on='id')

In [214]:
green_skills_data = green_skills_data.merge(created_data[['id', 'quarter']], how='inner', left_on='job_id', right_on='id')

In [299]:
print(skills_data['id'].n_unique())
print(skills_data_time['id'].n_unique())

5277868
5277868


In [302]:
skills_data_time.filter(pl.col('quarter')=='2023 Q3')['id'].n_unique()

4859

In [313]:
created_data[created_data['quarter']=='2023 Q3']['id'].nunique()

404907

In [303]:
skills_data_time.filter(pl.col('quarter')=='2023 Q2')['id'].n_unique()

430815

In [311]:
created_data[created_data['quarter']=='2023 Q2']['id'].nunique()

439415

## 1. Process data

In [47]:
# dont include job adverts with not many skills since the proportion of green skills might not be reflective
print(len(measures_df))
measures_df = measures_df[measures_df['NUM_SPLIT_ENTS'] > 5]
print(len(measures_df))

5967229
4766678


In [48]:
measures_df['PERC_GREEN'] = measures_df['PROP_GREEN']*100
measures_over_time = measures_df.groupby('quarter').agg(
    {'job_id': 'count',
     'NUM_SPLIT_ENTS': 'mean',
     'PERC_GREEN': 'mean',
     'GREEN TIMESHARE': 'mean',
     'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
    }).reset_index().rename(columns={
    'job_id': 'Number of job ads',
    'NUM_SPLIT_ENTS': 'Average number of skills',
    'PERC_GREEN': 'Average percentage of green skills',
    'GREEN TIMESHARE': 'Average percentage of time spent on green tasks',
    'INDUSTRY GHG PER UNIT EMISSIONS': 'Average GHG emissions of industry'
})
measures_over_time['quarter'] = measures_over_time['quarter'].astype(str)
measures_over_time['quarter'] = measures_over_time['quarter'].apply(lambda x: x.replace("Q", " Q"))

# There is incomplete data for these quarters too
measures_over_time = measures_over_time[~measures_over_time['quarter'].isin(['2024 Q4', '2020 Q4'])] 

measures_over_time_melt = measures_over_time.melt(id_vars='quarter')
measures_over_time_melt.head(2)

,quarter,variable,value
0,2021 Q1,Number of job ads,317693.0
1,2021 Q2,Number of job ads,368698.0


## Do the same per SOC too
- the big occs
- the most green
- the least green

In [149]:
soc_av_green_skills = measures_df.groupby('SOC_2020_EXT_name')['PERC_GREEN'].mean().reset_index()
top_green_skill_occs = soc_av_green_skills.sort_values(by='PERC_GREEN', ascending=False)[0:20]['SOC_2020_EXT_name'].to_list()

In [150]:
soc_av_ghg = measures_df.groupby('SOC_2020_EXT_name')['INDUSTRY GHG PER UNIT EMISSIONS'].mean().reset_index()
top_ghg = soc_av_ghg.sort_values(by='INDUSTRY GHG PER UNIT EMISSIONS', ascending=False)[0:20]['SOC_2020_EXT_name'].to_list()

In [136]:
count_soc = measures_df.groupby('SOC_2020_EXT_name')['job_id'].count().reset_index()
large_soc = count_soc[count_soc['job_id']>10000]['SOC_2020_EXT_name'].tolist()
len(large_soc)

102

In [156]:
# Include SOCs from a lot of job advs, or from SOCs of interest as long as they have over 100 job advs
socs_to_inc = large_soc + count_soc[((count_soc['SOC_2020_EXT_name'].isin(top_green_skill_occs+top_ghg)) & (count_soc['job_id']>100))]['SOC_2020_EXT_name'].to_list()

In [413]:
measures_df['job_id'].nunique()

4766678

In [157]:
measures_over_time_per_soc = measures_df[measures_df['SOC_2020_EXT_name'].isin(socs_to_inc)].groupby(['SOC_2020_EXT_name', 'quarter']).agg(
    {'job_id': 'count',
     'NUM_SPLIT_ENTS': 'mean',
     'PERC_GREEN': 'mean',
     'GREEN TIMESHARE': 'mean',
     'INDUSTRY GHG PER UNIT EMISSIONS': 'mean',
    }).reset_index().rename(columns={
    'job_id': 'Number of job ads',
    'NUM_SPLIT_ENTS': 'Average number of skills',
    'PERC_GREEN': 'Average percentage of green skills',
    'GREEN TIMESHARE': 'Average percentage of time spent on green tasks',
    'INDUSTRY GHG PER UNIT EMISSIONS': 'Average GHG emissions of industry'
})

measures_over_time_per_soc['quarter'] = measures_over_time_per_soc['quarter'].astype(str)
measures_over_time_per_soc['quarter'] = measures_over_time_per_soc['quarter'].apply(lambda x: x.replace("Q", " Q"))

# There is incomplete data for these quarters too
measures_over_time_per_soc = measures_over_time_per_soc[~measures_over_time_per_soc['quarter'].isin(['2024 Q4', '2020 Q4'])] 

measures_over_time_per_soc = measures_over_time_per_soc.melt(id_vars=['SOC_2020_EXT_name', 'quarter'])
measures_over_time_per_soc.head(2)

,SOC_2020_EXT_name,quarter,variable,value
0,Accounting clerks and bookkeepers,2021 Q1,Number of job ads,10670.0
1,Accounting clerks and bookkeepers,2021 Q2,Number of job ads,12517.0


In [158]:
measures_over_time_per_soc_pivot = measures_over_time_per_soc.pivot(
    index=['quarter', 'variable'], columns='SOC_2020_EXT_name', values='value').reset_index().rename_axis(columns=[None],axis=1)

In [159]:
main_metrics_over_time = measures_over_time_melt.rename(columns={'value': 'All adverts'}).merge(measures_over_time_per_soc_pivot, on=['quarter', 'variable']).round(3)

In [160]:
main_metrics_over_time.to_csv(f"{graph_dir}/over_time.csv")

In [410]:
main_metrics_over_time[main_metrics_over_time['variable']=='Number of job ads']['All adverts'].sum()

4654348.0

## Most common green and not-green skills

In [203]:
green_skill_id_2_name, full_skill_id_2_name = pg.read_process_taxonomies()

2024-12-03 09:32:50,389 - dap_prinz_green_jobs - INFO - Loading skills taxonomies
2024-12-03 09:32:50,475 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


### Get the most common 100 skills and cut the data down to this (to speed things up)

In [162]:
skills_data_time_pd = skills_data_time.to_pandas()

In [297]:
len(skills_data_time_pd)

74302772

In [225]:
from tqdm import tqdm

In [369]:
skills_data_time_pd['esco_name'] = skills_data_time_pd['esco_id'].map(full_skill_id_2_name)

In [370]:
all_df = pd.DataFrame()
for year, v in tqdm(skills_data_time_pd.groupby('quarter')):

    vv = v.groupby('esco_name')['id'].nunique().reset_index()
    df = vv.sort_values(by='id', ascending=False).reset_index(drop=True).reset_index().rename(
        columns={'index': 'Rank', 'id':'Number of job adverts'})
    df['Q'] = year
    df['Percentage of job adverts for this Q'] = df['Number of job adverts']*100/v['id'].nunique()
    all_df = pd.concat([all_df, df])
all_df['Skill Type']='All skills'

100%|████████████████████████████████████████████████████████████████████████████████████████| 17/17 [00:42<00:00,  2.48s/it]


In [371]:
green_skills_data['esco_name'] = green_skills_data['extracted_green_skill_id'].map(green_skill_id_2_name)

In [372]:
green_all_df = pd.DataFrame()
for year, v in green_skills_data.groupby('quarter'):

    vv = v.groupby('esco_name')['job_id'].nunique().reset_index()
    df = vv.sort_values(by='job_id', ascending=False).reset_index(drop=True).reset_index().rename(
        columns={'index': 'Rank', 'job_id':'Number of job adverts'})
    df['Q'] = year
    df['Percentage of job adverts for this Q'] = df['Number of job adverts']*100/v['id'].nunique()
    green_all_df = pd.concat([green_all_df, df])
green_all_df['Skill Type']='Green skills'

In [376]:
green_all_df.head(2)

,Rank,esco_name,Number of job adverts,Q,Percentage of job adverts for this Q,Skill Type
0,0,engage others in environment friendly behaviours,69,2020 Q4,8.070175,Green skills
1,1,follow health and safety procedures in constru...,56,2020 Q4,6.549708,Green skills


In [374]:
both_skills = pd.concat([all_df, green_all_df])

In [375]:
both_skills.head(2)

,Rank,esco_name,Number of job adverts,Q,Percentage of job adverts for this Q,Skill Type
0,0,demonstrate enthusiasm,2707,2020 Q4,18.662530,All skills
1,1,communication,2704,2020 Q4,18.641848,All skills


In [429]:
before = all_df[all_df['Q']=='2021 Q3'][['esco_name', 'Number of job adverts', 'Percentage of job adverts for this Q']]
after = all_df[all_df['Q']=='2024 Q3'][['esco_name', 'Number of job adverts', 'Percentage of job adverts for this Q']]

In [430]:
print(len(before))
print(before['esco_name'].nunique())

print(len(after))
print(after['esco_name'].nunique())

10394
10394
10155
10155


In [431]:
merged_before_after = before.merge(after, on= 'esco_name', suffixes=('_2021_q3', '_2024_q3'))
merged_before_after['perc_diff'] = merged_before_after['Percentage of job adverts for this Q_2021_q3'] - merged_before_after['Percentage of job adverts for this Q_2024_q3']

In [432]:
merged_before_after.to_csv(f'{graph_dir}/skill_differences_3_years.csv', index=False)

In [433]:
# Which skills are in lots of job adverts
notgreen_counts = all_df.groupby('esco_name')['Number of job adverts'].sum().reset_index()
notgreen_top = notgreen_counts.sort_values(by=['Number of job adverts'], ascending=False)[0:10]['esco_name'].to_list()

green_counts = green_all_df.groupby('esco_name')['Number of job adverts'].sum().reset_index()
green_top = green_counts.sort_values(by=['Number of job adverts'], ascending=False)[0:10]['esco_name'].to_list()


In [434]:
all_df_pivot = (all_df.pivot_table(index=['Q', 'Skill Type'], 
                      columns='esco_name', 
                      values='Percentage of job adverts for this Q', 
                      aggfunc='first')
         .reset_index()
         .rename_axis(None, axis=1))

green_df_pivot = (green_all_df.pivot_table(index=['Q', 'Skill Type'], 
                      columns='esco_name', 
                      values='Percentage of job adverts for this Q', 
                      aggfunc='first')
         .reset_index()
         .rename_axis(None, axis=1))

In [435]:
final_data = pd.concat([
    green_df_pivot[['Q', 'Skill Type']+green_top],
    all_df_pivot[['Q', 'Skill Type']+notgreen_top],
    
]
)
# Take out weird years (not many skills in 2023 Q3) and incomplete years
final_data = final_data[~final_data['Q'].isin(['2020 Q4', '2024 Q4', '2023 Q3'])]


final_data.to_csv(f'{graph_dir}/skill_rank_time.csv', index=False)

In [380]:
## How many job ads minimum

In [436]:
all_df.head(2)

,Rank,esco_name,Number of job adverts,Q,Percentage of job adverts for this Q,Skill Type
0,0,demonstrate enthusiasm,2707,2020 Q4,18.662530,All skills
1,1,communication,2704,2020 Q4,18.641848,All skills


In [437]:
qs = all_df.groupby('Q')['Number of job adverts'].sum()
print(qs.min())
print(qs.max())

61039
6463980


In [438]:
qs = green_all_df.groupby('Q')['Number of job adverts'].sum()
print(qs.min())
print(qs.max())

1093
44319


In [439]:
all_df[((~all_df['Q'].isin(
    ['2020 Q4', '2024 Q4', '2023 Q3'])) & (all_df['esco_name'].isin(notgreen_top)) & (all_df['Number of job adverts']<5000))]

,Rank,esco_name,Number of job adverts,Q,Percentage of job adverts for this Q,Skill Type
3,3,attend to detail,4953,2023 Q1,13.369504,All skills
4,4,leading and motivating,4929,2023 Q1,13.304721,All skills
5,5,"communication, collaboration and creativity",4883,2023 Q1,13.180554,All skills
6,6,coordinating activities with others,4587,2023 Q1,12.381569,All skills
7,7,coaching and mentoring,4509,2023 Q1,12.171026,All skills
8,8,developing professional relationships or networks,4407,2023 Q1,11.895700,All skills
9,9,management skills,4182,2023 Q1,11.288363,All skills


In [440]:
green_all_df[((
    ~green_all_df['Q'].isin(['2020 Q4', '2024 Q4', '2023 Q3'])) & (
        green_all_df['esco_name'].isin(green_top)) & (green_all_df['Number of job adverts']<100))]

,Rank,esco_name,Number of job adverts,Q,Percentage of job adverts for this Q,Skill Type
3,3,manage habitats,99,2023 Q1,4.227156,Green skills
4,4,promote innovative infrastructure design,91,2023 Q1,3.885568,Green skills
5,5,ensure compliance with environmental legislation,86,2023 Q1,3.672075,Green skills
6,6,perform cleaning activities in an environmenta...,83,2023 Q1,3.543980,Green skills
7,7,environmental engineering,60,2023 Q1,2.561913,Green skills
11,11,engage others in environment friendly behaviours,55,2023 Q1,2.348420,Green skills
12,12,waste management,51,2023 Q1,2.177626,Green skills
